# Pokémon primary type (`type_1`) classification

This notebook keeps the original target **`type_1`**. It compares a majority-class baseline, scaled logistic regression, random forest, and extra trees, then tunes Extra Trees using stratified cross-validation. The test set is held out until final evaluation.

The dataset link must be publicly accessible to Google Drive download tools. Scores are not guaranteed to exceed an earlier run: primary type is not fully determined by the available stats, and rare types make the task difficult.

In [ ]:
# In Google Colab, install gdown to fetch the shared CSV from Google Drive.
%pip -q install gdown
import gdown
import numpy as np
import pandas as pd

FILE_ID = '12C1MY0dcLSouDrRCVkm3DJ2_yDQxe0xC'
CSV_PATH = 'pokemon_data.csv'
downloaded = gdown.download(id=FILE_ID, output=CSV_PATH, quiet=False)
if downloaded is None:
    raise RuntimeError('Download failed. Confirm the Drive file is shared with anyone who has the link, then retry.')
df = pd.read_csv(CSV_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## Check data and class balance

We preserve `type_1` as the label, convert missing `type_2` values to an explicit category (no secondary type), and impute missing numeric values inside each model pipeline. `name` is left out because it is a unique identifier-like text field. `id` is retained as a numeric predictor; it can encode generation/order patterns, so interpret results as random-row holdout performance rather than performance on future generations.

In [ ]:
TARGET = 'type_1'  # Keep the original target unchanged.
if TARGET not in df.columns:
    raise ValueError(f'Required target column {TARGET!r} is missing. Found: {df.columns.tolist()}')
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
df['type_2'] = df['type_2'].fillna('none').astype(str).str.strip().str.lower()

print('Rows:', len(df), '| Classes:', df[TARGET].nunique())
display(df[TARGET].value_counts().rename('count').to_frame())
display(df.isna().sum().rename('missing values').to_frame())

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X = df.drop(columns=[TARGET, 'name'], errors='ignore')
y = df[TARGET]
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
print('Predictors:', X.columns.tolist())
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

# Stratification keeps the rare classes represented in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} rows; test: {len(X_test)} rows')

## Preprocessing and cross-validated model comparison

Preprocessing is fitted on each training fold only, preventing test-fold information from leaking into imputation, scaling, or category encoding. Macro-F1 gives each type equal weight; accuracy and balanced accuracy are shown as additional metrics.

In [ ]:
numeric_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
numeric_trees = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

def make_preprocessor(scale_numeric=False):
    numeric_transform = numeric_scaled if scale_numeric else numeric_trees
    return ColumnTransformer([
        ('numeric', numeric_transform, numeric_features),
        ('categorical', categorical, categorical_features)
    ], remainder='drop')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Most frequent baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': Pipeline([('prep', make_preprocessor(scale_numeric=True)),
                                  ('model', LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0))]),
    'Random forest': Pipeline([('prep', make_preprocessor()),
                               ('model', RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                                               class_weight='balanced_subsample', random_state=42, n_jobs=1))]),
    'Extra trees': Pipeline([('prep', make_preprocessor()),
                            ('model', ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2,
                                                         class_weight='balanced', random_state=42, n_jobs=1))])
}
scoring = {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy', 'f1_macro': 'f1_macro'}
rows = []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        'model': name,
        'CV accuracy': scores['test_accuracy'].mean(),
        'CV balanced accuracy': scores['test_balanced_accuracy'].mean(),
        'CV macro-F1': scores['test_f1_macro'].mean(),
        'macro-F1 std': scores['test_f1_macro'].std()
    })
comparison = pd.DataFrame(rows).sort_values('CV macro-F1', ascending=False).reset_index(drop=True)
display(comparison.style.format({c: '{:.3f}' for c in comparison.select_dtypes('number').columns}))

## Tune Extra Trees

Search parameters using only the training split and stratified folds. The objective is macro-F1, so a model is not rewarded solely for doing well on the most common types.

In [ ]:
search_pipeline = Pipeline([
    ('prep', make_preprocessor()),
    ('model', ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=1))
])
param_distributions = {
    'model__n_estimators': [300, 500, 800],
    'model__max_depth': [None, 15, 25, 40],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 0.7, 1.0]
}
search = RandomizedSearchCV(
    search_pipeline, param_distributions=param_distributions, n_iter=20,
    scoring='f1_macro', cv=cv, n_jobs=-1, random_state=42, verbose=1, refit=True
)
search.fit(X_train, y_train)
print('Best CV macro-F1:', round(search.best_score_, 4))
print('Best parameters:', search.best_params_)
best_model = search.best_estimator_

## Final holdout evaluation

Use the test split once for the tuned model. Compare its results with the majority baseline. A low score is still meaningful: it indicates that the available columns may not contain enough information to reliably infer every primary type.

In [ ]:
test_predictions = best_model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
final_scores = pd.DataFrame([
    {'model': 'Majority baseline', 'accuracy': accuracy_score(y_test, baseline_predictions),
     'balanced accuracy': balanced_accuracy_score(y_test, baseline_predictions),
     'macro-F1': f1_score(y_test, baseline_predictions, average='macro', zero_division=0)},
    {'model': 'Tuned Extra Trees', 'accuracy': accuracy_score(y_test, test_predictions),
     'balanced accuracy': balanced_accuracy_score(y_test, test_predictions),
     'macro-F1': f1_score(y_test, test_predictions, average='macro', zero_division=0)}
])
display(final_scores.style.format({c: '{:.3f}' for c in ['accuracy', 'balanced accuracy', 'macro-F1']}))
print(classification_report(y_test, test_predictions, labels=sorted(y.unique()), zero_division=0))

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=sorted(y.unique()), xticks_rotation=45,
    cmap='Blues', colorbar=False, ax=ax
)
ax.set_title('Tuned Extra Trees — held-out test set')
plt.tight_layout()
plt.show()

## Optional: inspect model errors

This table helps identify which types are commonly confused. Do not use the test errors to keep tuning against the same test split; use cross-validation or a new holdout for further experiments.

In [ ]:
errors = X_test.copy()
errors['actual_type_1'] = y_test
errors['predicted_type_1'] = test_predictions
display(errors.loc[errors['actual_type_1'] != errors['predicted_type_1']].head(25))